In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

<details>
<summary><b>إصدارات الحزم</b></summary>

تم تطوير الكود في هذه الصفحة باستخدام المتطلبات التالية.
نوصي باستخدام هذه الإصدارات أو أحدث منها.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
بالإضافة إلى [تصوير التعليمات على الدائرة](/guides/visualize-circuits)، قد ترغب في تصوير جدولة الدائرة باستخدام طريقة Qiskit [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer). يمكن أن يساعدك هذا التصوير مثلاً في اكتشاف وقت التوقف على الـ Qubits بسرعة. غير أن هذه الطريقة لا تُعيد نتائج دقيقة للدوائر الديناميكية. لتصوير جدولة الدوائر الديناميكية، استخدم طريقة `draw_circuit_schedule_timing` كما هو موضح في قسم [دعم Qiskit Runtime](#qr-support).

## أمثلة

لتصوير برنامج دائرة مجدوَلة، يمكنك استدعاء هذه الدالة مع مجموعة من وسائط التحكم. يمكن تعديل معظم مظهر صورة الإخراج عبر ورقة أنماط، لكن ذلك ليس إلزامياً.

### الرسم باستخدام ورقة الأنماط الافتراضية

In [1]:
from qiskit import QuantumCircuit
from qiskit.visualization.timeline import draw
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

backend = GenericBackendV2(5)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

draw(isa_circuit, target=backend.target)

<Image src="../docs/images/guides/visualize-circuit-timing/extracted-outputs/5b21b8c1-0.svg" alt="Output of the previous code cell" />

![مخرجات خلية الكود السابقة](../docs/images/guides/visualize-circuit-timing/extracted-outputs/5b21b8c1-0.svg)

### الرسم باستخدام ورقة أنماط مناسبة لتصحيح البرامج

In [2]:
from qiskit import QuantumCircuit
from qiskit.visualization.timeline import draw, IQXDebugging
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

backend = GenericBackendV2(5)
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)
draw(isa_circuit, style=IQXDebugging(), target=backend.target)

<Image src="../docs/images/guides/visualize-circuit-timing/extracted-outputs/907dc46c-0.svg" alt="Output of the previous code cell" />

![مخرجات خلية الكود السابقة](../docs/images/guides/visualize-circuit-timing/extracted-outputs/907dc46c-0.svg)

يمكنك إنشاء دوال مُولِّد أو تخطيط مخصصة وتحديث ورقة أنماط موجودة بتلك الدوال. بهذه الطريقة يمكنك التحكم في معظم مظهر صورة الإخراج دون تعديل قاعدة الكود الخاصة بمُرسِم الدائرة المجدوَلة. راجع مرجع API لـ [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer) لمزيد من الأمثلة.
<span id="qr-support"></span>

## دعم Qiskit Runtime

بينما يُعدّ مُرسِم الجدول الزمني المدمج في Qiskit مفيداً للدوائر الثابتة، فإنه قد لا يعكس بدقة توقيت [الدوائر الديناميكية](/guides/classical-feedforward-and-control-flow) بسبب العمليات الضمنية مثل البث وتحديد الفروع. كجزء من دعم الدوائر الديناميكية، يُعيد Qiskit Runtime معلومات توقيت الدائرة الدقيقة داخل نتائج المهمة عند الطلب.

> **Note:** - هذه دالة تجريبية، وهي في وضع إصدار معاينة وبالتالي قد تتغير.
> - تنطبق هذه الدالة فقط على مهام Qiskit Runtime Sampler.
> - على الرغم من أن إجمالي وقت الدائرة يُعاد في بيانات التعريف "compilation"، إلا أن هذا ليس الوقت المُستخدَم للفوترة (الوقت الكمومي).

### تفعيل استرجاع بيانات التوقيت

لتفعيل استرجاع بيانات التوقيت، اضبط علامة `scheduler_timing` التجريبية على `True` عند تشغيل مهمة primitive.

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

sampler = SamplerV2(backend)
sampler.options.experimental = {
    "execution": {
        "scheduler_timing": True,
    },
}

sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

### الوصول إلى بيانات توقيت الدائرة
عند الطلب، تُعاد بيانات توقيت الدائرة لكل PUB في بيانات تعريف نتيجة المهمة، تحت `["compilation"]["scheduler_timing"]["timing"]`. يحتوي هذا الحقل على معلومات التوقيت الخام. لعرض معلومات التوقيت، استخدم أداة التصوير المدمجة كما هو موضح في قسم [تصوير التوقيتات](#visualize-timings).

استخدم الكود التالي للوصول إلى بيانات توقيت الدائرة لأول PUB:

In [4]:
job_result = sampler_job.result()
circuit_schedule = job_result[0].metadata["compilation"]["scheduler_timing"]
circuit_schedule_timing = circuit_schedule["timing"]

#### فهم بيانات التوقيت الخام
بينما يُعدّ تصوير بيانات توقيت الدائرة باستخدام طريقة `draw_circuit_schedule_timing` حالة الاستخدام الأكثر شيوعاً، قد يكون من المفيد فهم بنية بيانات التوقيت الخام المُعادة. قد يساعدك ذلك مثلاً على استخراج المعلومات برمجياً.

بيانات التوقيت المُعادة في `["compilation"]["scheduler_timing"]["timing"]` هي قائمة من السلاسل النصية. كل سلسلة تمثل تعليمة واحدة على قناة ما، وتكون مفصولة بفواصل إلى أنواع البيانات التالية:

- `Branch` - يُحدد ما إذا كانت التعليمة في تدفق تحكم (then / else) أو فرع رئيسي.
- `Instruction` - الـ Gate والـ Qubit الذي ستعمل عليه.
- `Channel` - القناة المُعيَّنة للتعليمة. يمكن أن تكون إحدى التالي:
      - `Qubit x` - قناة القيادة لـ Qubit _x_.
      - `AWGRx_y` (قراءة مولّد موجة اعتباطي) - تُستخدم بواسطة قنوات القراءة للتواصل عند قياس الـ Qubits. تقابل وسيطا _x_ و _y_ معرّف جهاز القراءة ورقم الـ Qubit على التوالي.
- `T0` - وقت بدء التعليمة ضمن الجدول الكامل.
- `Duration` - مدة التعليمة بوحدات _dt_ ثانية، حيث 1 dt = 1 دورة جدولة. يمكنك إيجاد قيمة `dt` للـ Backend باستخدام [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt).
- `Pulse` - نوع عملية النبضة المُستخدَمة.

مثال:

In [5]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

# Create a figure from the metadata
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule_timing,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

<span id="visualize-timings"></span>
### تصوير التوقيتات
مع `qiskit-ibm-runtime` الإصدار v0.43.0 أو أحدث، يمكنك تصوير توقيتات الدوائر. لتصوير التوقيتات، تحتاج أولاً إلى تحويل بيانات تعريف النتيجة إلى `fig` باستخدام [طريقة `draw_circuit_schedule_timing`.](https://github.com/Qiskit/qiskit-ibm-runtime/blob/3d1bf1e1d49e5123841639fce259859c90ce9314/qiskit_ibm_runtime/visualization/draw_circuit_schedule_timings.py#L26) تُعيد هذه الطريقة شكل `plotly`، يمكنك عرضه مباشرة أو حفظه في ملف أو كليهما. لمزيد من المعلومات حول أوامر `plotly` المُستخدَمة، راجع [`fig.show()`](https://plotly.com/python-api-reference/generated/plotly.io.show.html) و [`fig.write_image("<path.format>")`](https://plotly.com/python-api-reference/generated/plotly.io.write_image.html).

In [6]:
from qiskit_ibm_runtime import SamplerV2, QiskitRuntimeService
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Create a dynamic circuit

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
qc = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

qc.h(q0)
qc.measure(q0, c0)
with qc.if_test((c0, 1)):
    qc.x(q0)
qc.measure(q0, c0)

# Convert to an ISA circuit for the given backend

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

# Generate samplers for backend targets
sampler = SamplerV2(backend)
sampler.options.experimental = {"execution": {"scheduler_timing": True}}

# Submit jobs
sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

print(
    f">>> {' Job ID:':<10}  {sampler_job.job_id()} ({sampler_job.status()})"
)

>>>  Job ID:    d5kk3cn853es738e01dg (DONE)


![تمرير المؤشر فوق المخرجات يعرض معلومات مثل وقت البدء والانتهاء والمدة.](../docs/images/guides/visualize-circuit-timing/image_1.svg 'مثال على شكل مُولَّد')
#### فهم الشكل المُولَّد
تنقل صورة مخرجات بيانات توقيت الدائرة التي يُنتجها `draw_circuit_schedule_timing` المعلومات التالية:

- المحور X هو الوقت بوحدات _dt_ ثانية، حيث 1 dt = 1 دورة جدولة. يمكنك إيجاد قيمة `dt` للـ Backend باستخدام [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt).
- المحور Y هو القناة (فكّر في القنوات كأدوات تُصدر نبضات).
    - `Receive channel` - القناة الوحيدة التي ليست أداة بحد ذاتها. إنها تعليمة تُشغَّل على جميع القنوات التي تشارك في إجراء اتصال مع الـ Hub في تلك اللحظة.
    - `Qubit x` - قناة القيادة لـ Qubit x.
    - `AWGRx_y` (قراءة مولّد موجة اعتباطي) - تُستخدم بواسطة قنوات القراءة للتواصل عند قياس الـ Qubits. تقابل وسيطا _x_ و _y_ معرّف جهاز القراءة ورقم الـ Qubit على التوالي.
    - `Hub` - يتحكم في البث.

بالإضافة إلى ذلك، لكل تعليمة صيغة *X_Y*، حيث *X* هو اسم التعليمة و*Y* هو نوع النبضة. يُطبّق نوع `play` نبضات التحكم، بينما يُسجِّل `capture` حالة الـ Qubit. يمكنك أيضاً التمرير فوق كل تعليمة للحصول على مزيد من التفاصيل. على سبيل المثال، يُظهر الشكل السابق نبضة تحكم لـ Gate X مُطبَّقة على Qubit 10 عند 1161 dt.
### مثال شامل من البداية إلى النهاية
يُوضح هذا المثال كيفية تفعيل الخيار، والحصول على معلومات توقيت الدائرة من البيانات الوصفية، وعرضها كصورة.

أولاً، قم بإعداد البيئة وتعريف الدوائر وتحويلها إلى دوائر ISA، وتعريف المهام وتشغيلها.

In [7]:
# Get the circuit schedule timing
result[0].metadata["compilation"]["scheduler_timing"]["timing"]

'main,rz_0,Qubit 0,929,0,shift_phase\nmain,sx_0,Qubit 0,929,8,play\nmain,sx_0,Qubit 0,933,0,shift_phase\nmain,rz_0,Qubit 0,937,0,shift_phase\nmain,barrier,Qubit 0,937,0,barrier\nmain,barrier,Qubit 0,937,0,barrier\nmain,barrier,Qubit 1,937,0,barrier\nmain,barrier,Qubit 2,937,0,barrier\nmain,barrier,Qubit 3,937,0,barrier\nmain,barrier,Qubit 4,937,0,barrier\nmain,barrier,Qubit 5,937,0,barrier\nmain,barrier,Qubit 6,937,0,barrier\nmain,barrier,Qubit 7,937,0,barrier\nmain,barrier,Qubit 8,937,0,barrier\nmain,barrier,Qubit 9,937,0,barrier\nmain,barrier,Qubit 10,937,0,barrier\nmain,barrier,Qubit 11,937,0,barrier\nmain,barrier,Qubit 12,937,0,barrier\nmain,barrier,Qubit 13,937,0,barrier\nmain,barrier,Qubit 14,937,0,barrier\nmain,barrier,Qubit 15,937,0,barrier\nmain,barrier,Qubit 16,937,0,barrier\nmain,barrier,Qubit 17,937,0,barrier\nmain,barrier,Qubit 18,937,0,barrier\nmain,barrier,Qubit 19,937,0,barrier\nmain,barrier,Qubit 20,937,0,barrier\nmain,barrier,Qubit 21,937,0,barrier\nmain,barrier,Qubit

Finally, you can visualize and save the timing:

In [8]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

circuit_schedule = result[0].metadata["compilation"]["scheduler_timing"][
    "timing"
]
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

بعد ذلك، احصل على توقيت جدول الدائرة: